In [ ]:

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-xpveosfq/unsloth_3e5f4b4d09e843d3bcae216504f4805f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-xpveosfq/unsloth_3e5f4b4d09e843d3bcae216504f4805f
  Resolved https://github.com/unslothai/unsloth.git to commit 5fa8683b27333111657e1976166fabce7e4abed5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.8 MB/s eta 0:00:0

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# LoRA adapters add karna (Trainable parameters)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [9]:
from datasets import load_dataset

# Dataset load karna
dataset = load_dataset("json", data_files="train_llama3.jsonl", split="train")

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = f"### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n{output}"
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)
print("Dataset loaded and formatted successfully!")


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Dataset loaded and formatted successfully!


In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # 60 steps tak train hoga fast results ke liye
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

print("Training cycle started...")
trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/25000 [00:00<?, ? examples/s]

Training cycle started...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.741474
2,2.789019
3,2.656944
4,2.494534
5,2.131608
6,1.697775
7,1.525267
8,1.052473
9,0.799103
10,0.719179


TrainOutput(global_step=60, training_loss=0.6288642639915148, metrics={'train_runtime': 227.8132, 'train_samples_per_second': 2.107, 'train_steps_per_second': 0.263, 'total_flos': 1941024188055552.0, 'train_loss': 0.6288642639915148, 'epoch': 0.0192})

In [14]:
test_cases = [
    # --- 5 LEGITIMATE CASES ---
    "Amount: $45.0, Product Code: W, Card Type: visa debit, Address Code: 120.0, Distance: 1.0",
    "Amount: $5.0, Product Code: W, Card Type: mastercard debit, Address Code: 110.0, Distance: 0.0",
    "Amount: $15.5, Product Code: H, Card Type: visa credit, Address Code: 150.0, Distance: 0.0",
    "Amount: $120.0, Product Code: W, Card Type: mastercard credit, Address Code: 204.0, Distance: 2.5",
    "Amount: $75.0, Product Code: W, Card Type: visa debit, Address Code: 181.0, Distance: 3.0",

    # --- 5 SUSPICIOUS CASES ---
    "Amount: $2500.0, Product Code: R, Card Type: mastercard credit, Address Code: 450.0, Distance: 4000.0",
    "Amount: $1000.0, Product Code: W, Card Type: discover credit, Address Code: 350.0, Distance: 500.0",
    "Amount: $3000.0, Product Code: R, Card Type: visa credit, Address Code: 500.0, Distance: 200.0",
    "Amount: $600.0, Product Code: W, Card Type: mastercard credit, Address Code: 299.0, Distance: 950.0",
    "Amount: $5000.0, Product Code: H, Card Type: visa credit, Address Code: 600.0, Distance: 1500.0"
]

print("\n" + "="*50)
print("?? RUNNING 10-SAMPLE BATCH TEST ??")
print("="*50)

for i, test_transaction in enumerate(test_cases):
    prompt = f"### Instruction:\nAnalyze the following transaction details and determine if it is likely to be fraudulent ('Fraud') or legitimate ('Legitimate').\n\n### Input:\n{test_transaction}\n\n### Response:\n"
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # Fast Inference
    outputs = model.generate(**inputs, max_new_tokens = 16)
    prediction = tokenizer.batch_decode(outputs)[0].split('### Response:')[1].strip()

    label = "SAFE" if i < 5 else "SUSP"
    print(f"[{label}] Test {i+1}: Result -> {prediction}")

print("\n" + "="*50)


Both `max_new_tokens` (=16) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



?? RUNNING 10-SAMPLE BATCH TEST ??


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=16) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[SAFE] Test 1: Result -> Fraud

### Input:
Amount: $35.0, Product Code:


Both `max_new_tokens` (=16) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[SUSP] Test 7: Result -> Legitimate

### Details:
Legitimate transaction details: Amount: $1000


Both `max_new_tokens` (=16) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[SUSP] Test 8: Result -> Legitimate

### Details:
Legitimate ('Legitimate')<|end_of_text|>


Both `max_new_tokens` (=16) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[SUSP] Test 9: Result -> Legitimate

### Details:
Legitimate ('Legitimate')<|end_of_text|>
[SUSP] Test 10: Result -> Fraud

### Details:
Fraud: Amount: $5000.0



In [15]:

model.save_pretrained("llama3_fraud_model_final")
tokenizer.save_pretrained("llama3_fraud_model_final")


!zip -r llama3_fraud_model.zip llama3_fraud_model_final

print("Model saved and Zipped! Aap left sidebar se llama3_fraud_model.zip download kar sakte hain.")


  adding: llama3_fraud_model_final/ (stored 0%)
  adding: llama3_fraud_model_final/README.md (deflated 65%)
  adding: llama3_fraud_model_final/adapter_config.json (deflated 58%)
  adding: llama3_fraud_model_final/tokenizer.json (deflated 85%)
  adding: llama3_fraud_model_final/adapter_model.safetensors (deflated 8%)
  adding: llama3_fraud_model_final/tokenizer_config.json (deflated 45%)
Model saved and Zipped! Aap left sidebar se llama3_fraud_model.zip download kar sakte hain.


In [16]:
from google.colab import drive
import os


drive.mount('/content/drive')

save_path = "/content/drive/My Drive/Llama3_Fraud_Model"
if not os.path.exists(save_path):
    os.makedirs(save_path)

!cp -r llama3_fraud_model_final/* "/content/drive/My Drive/Llama3_Fraud_Model/"

print(f"Success! Aapka model Google Drive mein yahan save ho gaya hai: {save_path}")


Mounted at /content/drive
Success! Aapka model Google Drive mein yahan save ho gaya hai: /content/drive/My Drive/Llama3_Fraud_Model
